# Haiku Multiple Instance Learning

**Project Name:** Haiku (renamed from Haiku)

## Purpose
- Publication-ready notebook for reproducible training/evaluation.

## Notes
- Paths and checkpoints may be environment-specific.
- Run cells top-to-bottom and set your config paths first.


In [ ]:
import sys
from pathlib import Path

HAIKU_ROOT = Path('/home/yancui/Haiku')
if str(HAIKU_ROOT / 'src') not in sys.path:

from haiku.notebook_utils import setup_notebook, seed_everything
setup_notebook(project_root='/home/yancui/Haiku')
seed_everything(42)


In [ ]:
import hydra
from omegaconf import DictConfig, OmegaConf
import os
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm import tqdm
from transformers import BertTokenizer
import pandas as pd
import numpy as np
import torch.nn.functional as F
import json
import sys


#target_patches_id = np.load('/home/yancui/Haiku/checkpoints/Trimodal_20250905-2158_full_trainset/holdout_patch_id.npy')

codex_embedding = torch.load('/home/yancui/Haiku/res_embdding_220/new_codex_embedding.pt')

virtual_codex_embedding = torch.load('/home/yancui/Haiku/res_embdding_220/new_virtual_codex_embedding.pt')

region_label = torch.load('/home/yancui/Haiku/res_embdding_220/new_region_label.pt')

print(codex_embedding.shape)

sample_ids = list(json.load(open('/home/yancui/Haiku/overlap_samples_new.json')).keys())

ref_ids = sorted(sample_ids)

# Build bags of embeddings for each patient
# The mapping from region_label[i] -> ref_ids[region_label[i]] will give you a region/acquisition ID
patient_he_bags = dict()        # patient_id: [he_embedding_tensors]
patient_codex_bags = dict()  # patient_id: [codex_embedding_tensors]
patient_virt_bags = dict()   # patient_id: [virtual_codex_embedding_tensors]
patient_musk_bags = dict()   # patient_id: [musk_embedding_tensors]
patient_labels = dict()
patient_concat_bags = dict()    # patient_id: label (list or scalar per patient, depending)

for idx, region_l in enumerate(region_label):
    # Lookup the patient_id for this region/acquisition

    region_id = ref_ids[region_l]

    # Group embeddings by patient
    #patient_he_bags.setdefault(region_id, []).append(he_embedding[idx])
    patient_codex_bags.setdefault(region_id, []).append(codex_embedding[idx])
    patient_virt_bags.setdefault(region_id, []).append(virtual_codex_embedding[idx])
    #patient_musk_bags.setdefault(region_id, []).append(musk_embedding[idx])
    #patient_concat_bags.setdefault(region_id, []).append(torch.cat([he_embedding[idx], codex_embedding[idx]], dim=0))

    # Optionally: Also maintain region-labels or per-patient labels
    # Here, defaulting to using the region's integer label.
    patient_labels.setdefault(region_id, []).append(region_label[idx])

# Optionally, if you want a "single label per patient" for classification (e.g. majority, first, or a function):
# Example (majority label per patient):
from collections import Counter
patient_majority_labels = {
    pid: Counter(lbls).most_common(1)[0][0]
    for pid, lbls in patient_labels.items()
}

# patient_bags, patient_codex_bags, patient d_virt_bags, patient_musk_bags now each map a patient_id -> list of embeddings as bags
# patient_labels: patient_id -> list of integer region label per patch, or use patient_majority_labels for (patient_id -> majority label)


In [ ]:
import os
import pandas as pd

csv_list = ['/home/yancui/Haiku/Lymphoma_response-to-RCHOP (1).csv',
'/home/yancui/Haiku/Melanoma_response-to-immunotherapy (1).csv',
'/home/yancui/Haiku/CRC_terminal_survival.csv']


need_new_acq_ids = []

df_list = []

for i in csv_list:
    df = pd.read_csv(i)
    new_acq_ids = df['ACQUISITION_ID'].tolist()
    df_list.append(df)



In [ ]:
survial_length_dict = {}
survial_status_dict = { }
response_dict = {}
treatment_dict = {}

In [ ]:
survial_length_dict['1'] = 'survival'
survial_status_dict['1'] = 'survival_status'
response_dict['1'] = 'Response-binary'
treatment_dict['1'] = 'treatment'

survial_length_dict['2'] = 'FOLLOW UP (months)'
survial_status_dict['2'] = 'survival_status'
response_dict['2'] = 'Outcome-binary'
treatment_dict['2'] = 'treatment'

In [ ]:
# End-to-end Multiple-Instance Learning (MIL) with Cox loss:
# - Splits into train, val, *test* and reports test C-index & KM
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '7'

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from typing import Dict, List, Tuple
import random
import matplotlib.pyplot as plt
import pandas as pd

from lifelines import KaplanMeierFitter
from lifelines.utils import concordance_index
from lifelines.statistics import logrank_test

# ----------------------------
# 0) Reproducibility
# ----------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# ---------------------------------------
# 1) Expected input dicts (replace these)
# ---------------------------------------
# These three dicts + four embedding dicts (for four methods/models now):
# time_dict  : {bag_id: float_survival_time}
# event_dict : {bag_id: int_event (1=death, 0=censored)}
# baseline_embeddings : {bag_id: [np.ndarray (Din,), ...]}
# our_embeddings      : {bag_id: [np.ndarray (Din,), ...]}
# third_method_embeddings : {bag_id: [np.ndarray (Din,), ...]}
# fourth_method_embeddings: {bag_id: [np.ndarray (Din,), ...]}


index = 2

df = df_list[index]

region_labels_survival = {}
region_values_survival_status = {}

for region_id in sample_ids:
    if region_id in df['ACQUISITION_ID'].values:
        region_labels_survival[region_id] = df[df['ACQUISITION_ID'] == region_id][survial_length_dict[str(index)]].values
        region_values_survival_status[region_id] = df[df['ACQUISITION_ID'] == region_id][survial_status_dict[str(index)]].values

interaction_region_ids = set(region_labels_survival.keys()) & set(region_values_survival_status.keys())

# If you want DataFrames/labels only for these interactions:
region_labels_survival_interaction = {rid: float(region_labels_survival[rid]) for rid in interaction_region_ids}
region_values_survival_status_interaction = {rid: int(region_values_survival_status[rid] != 'Alive') for rid in interaction_region_ids}

time_dict = region_labels_survival_interaction
event_dict = region_values_survival_status_interaction
baseline_embeddings = patient_virt_bags
our_embeddings = patient_codex_bags

print(len(list(interaction_region_ids)))

In [ ]:
# ============================================================
# End-to-end MIL–Cox: 5-fold CV (Baseline vs Ours)
# - Trains per fold
# - Reports per-fold Test C-index
# - Saves metrics JSON
# - Plots C-index boxplot (TEST)
# - Saves per-fold KM curves + log-rank p-values (TEST)
# ============================================================
import os, json, random
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from typing import Dict, List
import matplotlib.pyplot as plt
import pandas as pd

from lifelines import KaplanMeierFitter
from lifelines.utils import concordance_index
from lifelines.statistics import logrank_test

from sklearn.model_selection import StratifiedKFold, train_test_split

# --------------------- Matplotlib: SVG text editable ---------------------
plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype']  = 42

# --------------------- Reproducibility ---------------------
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

device = "cuda" if torch.cuda.is_available() else "cpu"

# =====================================================================
# REQUIRED INPUTS (provide these in your environment before running!)
# =====================================================================
# time_dict            = {bag_id: float_survival_time}
# event_dict           = {bag_id: int_event (1=death, 0=censored)}
# baseline_embeddings  = {bag_id: [np.ndarray(D,), ...]}
# our_embeddings       = {bag_id: [np.ndarray(D,), ...]}

# =====================================================================
# Dataset & collate
# =====================================================================
def l2_normalize(x, axis=1, eps=1e-12):
    norm = np.linalg.norm(x, ord=2, axis=axis, keepdims=True)
    return x / np.clip(norm, a_min=eps, a_max=None)

class MILDataset(Dataset):
    def __init__(self, emb_dict: Dict[str, List[np.ndarray]],
                 time_dict: Dict[str, float],
                 event_dict: Dict[str, int],
                 bag_ids: List[str]):
        self.ids, self.bags, self.times, self.events = [], [], [], []
        for bid in bag_ids:
            if bid not in emb_dict or bid not in time_dict or bid not in event_dict:
                continue
            insts = emb_dict[bid]
            if len(insts) == 0:
                continue
            X = np.stack(insts, axis=0).astype(np.float32)
            X = l2_normalize(X, axis=1)
            self.ids.append(bid)
            self.bags.append(torch.from_numpy(X))
            self.times.append(float(time_dict[bid]))
            self.events.append(int(event_dict[bid]))
        if len(self.bags) == 0:
            raise ValueError("Empty MILDataset — check input dicts/split.")
        self.Din = self.bags[0].shape[1]

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        return (
            self.bags[i],
            torch.tensor(self.times[i], dtype=torch.float32),
            torch.tensor(self.events[i], dtype=torch.long),
            self.ids[i],
        )

def pad_collate(batch):
    bags, times, events, ids = zip(*batch)
    B = len(bags); Nmax = max(b.shape[0] for b in bags); D = bags[0].shape[1]
    X = torch.zeros(B, Nmax, D, dtype=torch.float32)
    M = torch.zeros(B, Nmax, dtype=torch.bool)
    for i, b in enumerate(bags):
        n = b.shape[0]
        X[i, :n] = b
        M[i, :n] = True
    t = torch.stack(times)
    e = torch.stack(events)
    return X, M, t, e, list(ids)

# =====================================================================
# MIL–Cox model & loss
# =====================================================================
def cox_partial_ll(risk: torch.Tensor, time: torch.Tensor, event: torch.Tensor):
    # Sort by descending time so risk sets are cumulative from largest time to smallest
    order = torch.argsort(time, descending=True)
    risk  = risk[order]
    event = event[order].float()
    log_cumsum_exp = torch.logcumsumexp(risk, dim=0)
    ll = event * (risk - log_cumsum_exp)
    denom = event.sum().clamp_min(1.0)
    return -ll.sum() / denom

class AttnPool(nn.Module):
    def __init__(self, d, hidden=128):
        super().__init__()
        self.V = nn.Linear(d, hidden)
        self.w = nn.Linear(hidden, 1, bias=False)

    def forward(self, H, mask):
        A = self.w(torch.tanh(self.V(H))).squeeze(-1)   # [B,N]
        A = A.masked_fill(~mask, float("-inf"))
        A = torch.softmax(A, dim=1)
        Z = torch.einsum("bn,bnd->bd", A, H)            # [B,d]
        return Z, A

class MILCox(nn.Module):
    def __init__(self, in_dim, embed_dim=64, pool="attn"):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, embed_dim), nn.LeakyReLU(),
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, embed_dim), nn.LeakyReLU(),
            nn.LayerNorm(embed_dim),
        )
        self.pool_type = pool
        self.pool = AttnPool(embed_dim) if pool == "attn" else None
        self.head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim), nn.LeakyReLU(),
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 1),
        )

    def forward(self, X, mask):
        H = self.encoder(X)
        if self.pool_type == "attn":
            Z, A = self.pool(H, mask)
        elif self.pool_type == "mean":
            Z = (H * mask.unsqueeze(-1)).sum(1) / (
                mask.sum(1, keepdim=True).clamp_min(1)
            ).to(H.dtype)
            A = None
        elif self.pool_type == "max":
            Hm = H.masked_fill(
                ~mask.unsqueeze(-1), torch.finfo(H.dtype).min / 2
            )
            Z = Hm.max(dim=1).values
            A = None
        else:
            raise ValueError("pool must be one of {'attn','mean','max'}")
        risk = self.head(Z).squeeze(-1)  # higher = worse
        return risk, A

# =====================================================================
# Train / predict
# =====================================================================
def train_end2end_milcox(emb_dict, time_dict, event_dict, train_ids, val_ids,
                         batch_size=64, max_epochs=10, pool="attn", lr=5e-4, wd=0.0,
                         embed_dim=64, device=device, seed=42):
    set_seed(seed)
    train_ds = MILDataset(emb_dict, time_dict, event_dict, train_ids)
    val_ds   = MILDataset(emb_dict, time_dict, event_dict, val_ids)
    in_dim = train_ds.Din

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  collate_fn=pad_collate)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, collate_fn=pad_collate)

    model = MILCox(in_dim=in_dim, embed_dim=embed_dim, pool=pool).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)

    for epoch in range(1, max_epochs+1):
        model.train()
        tot, n = 0.0, 0
        for X, M, T, E, _ in train_loader:
            X, M, T, E = X.to(device), M.to(device), T.to(device), E.to(device)
            risk, _ = model(X, M)
            loss = cox_partial_ll(risk, T, E)
            opt.zero_grad()
            loss.backward()
            opt.step()
            tot += loss.item() * X.size(0)
            n += X.size(0)

        # quick val C-index
        model.eval()
        vt, ve, vr = [], [], []
        with torch.no_grad():
            for X, M, T, E, _ in val_loader:
                r, _ = model(X.to(device), M.to(device))
                vt.append(T.numpy())
                ve.append(E.numpy())
                vr.append(r.cpu().numpy())
        vt = np.concatenate(vt); ve = np.concatenate(ve); vr = np.concatenate(vr)
        cidx = concordance_index(vt, -vr, ve)
        print(f"[Epoch {epoch:03d}] train_loss={tot/max(n,1):.4f}  val_C-index={cidx:.4f}")

    return model

def predict_df(model, emb_dict, time_dict, event_dict, ids, batch_size=256, device=device):
    ds = MILDataset(emb_dict, time_dict, event_dict, ids)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, collate_fn=pad_collate)
    out_ids, out_t, out_e, out_r = [], [], [], []
    model.eval()
    with torch.no_grad():
        for X, M, T, E, ID in loader:
            r, _ = model(X.to(device), M.to(device))
            print(E)
            out_ids += ID
            out_t   += T.tolist()
            out_e   += E.tolist()
            out_r   += r.cpu().numpy().tolist()
    return pd.DataFrame({"bag_id": out_ids, "time": out_t, "event": out_e, "risk": out_r})

# =====================================================================
# KM helpers (per-fold curves + log-rank p-value)
# =====================================================================
def km_curves_and_logrank(df: pd.DataFrame):
    """
    Given a test DataFrame with columns ['time','event','risk'],
    split by median risk → KM curves (Low/High) + log-rank p-value.
    Returns:
        curves: dict with keys 'Low', 'High', each → (time_array, surv_array)
        p_value: float
    """
    risk = df["risk"].values
    time = df["time"].values
    event = df["event"].values

    med = np.median(risk)
    grp = np.where(risk >= med, "High", "Low")

    curves = {}
    km = KaplanMeierFitter()
    for label in ["Low", "High"]:
        m = (grp == label)
        if m.sum() < 2:
            curves[label] = (np.array([0.0, 1.0]), np.array([1.0, 1.0]))
            continue
        km.fit(durations=time[m], event_observed=event[m], label=label)
        t = km.survival_function_.index.values.astype(float)
        s = km.survival_function_[label].values.astype(float)
        if t[0] > 0:
            t = np.insert(t, 0, 0.0)
            s = np.insert(s, 0, 1.0)
        curves[label] = (t, s)

    # log-rank test between Low vs High
    low_mask = (grp == "Low")
    high_mask = (grp == "High")
    if low_mask.sum() >= 2 and high_mask.sum() >= 2:
        res = logrank_test(
            time[low_mask], time[high_mask],
            event_observed_A=event[low_mask],
            event_observed_B=event[high_mask]
        )
        p_value = float(res.p_value)
    else:
        p_value = float("nan")

    return curves, p_value

# =====================================================================
# 5-fold CV driver (Baseline vs Ours)
# =====================================================================
def run_cv5_milcox(time_dict, event_dict, baseline_embeddings, our_embeddings,
                   out_dir="cox_cv5_outputs",
                   pool="attn", embed_dim=64, lr=5e-4, wd=0.0,
                   max_epochs=10, batch_size=64, seed=42):
    os.makedirs(out_dir, exist_ok=True)
    set_seed(seed)

    # Universe of usable bag ids
    common_ids = set(time_dict) & set(event_dict) & set(baseline_embeddings) & set(our_embeddings)
    all_ids = sorted(common_ids)
    if len(all_ids) < 5:
        raise ValueError("Not enough bags for 5-fold CV.")
    y_event = np.array([event_dict[i] for i in all_ids])
    y_time  = np.array([time_dict[i]  for i in all_ids])

    # Stratify by (event, coarse time quartile)
    q = np.quantile(y_time, [0.25, 0.5, 0.75])
    time_bucket = np.digitize(y_time, q)  # 0..3
    strata = y_event.astype(str) + "_" + time_bucket.astype(str)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

    base_cidx_folds, ours_cidx_folds = [], []
    base_tests_folds, ours_tests_folds = [], []

    # store per-fold KM p-values
    base_km_pvals, ours_km_pvals = [], []

    fold = 0
    for tr_idx, te_idx in skf.split(all_ids, strata):
        fold += 1
        print(f"\n===== Fold {fold}/5 =====")
        train_ids = [all_ids[i] for i in tr_idx]
        test_ids  = [all_ids[i] for i in te_idx]

        # Small validation from train for early feedback (10% stratified on event)
        y_tr_event = np.array([event_dict[i] for i in train_ids])
        if len(train_ids) >= 20 and len(np.unique(y_tr_event)) == 2:
            tr_ids, va_ids = train_test_split(
                train_ids,
                test_size=max(1, int(0.1 * len(train_ids))),
                random_state=seed,
                stratify=y_tr_event
            )
        else:
            tr_ids, va_ids = train_ids, test_ids  # fallback

        # ---- Train Baseline ----
        print(f"[Fold {fold}] Training Baseline ...")
        base_model = train_end2end_milcox(
            baseline_embeddings, time_dict, event_dict, tr_ids, va_ids,
            batch_size=batch_size, max_epochs=max_epochs,
            pool=pool, lr=lr, wd=wd, embed_dim=embed_dim, device=device, seed=seed
        )
        base_test = predict_df(base_model, baseline_embeddings, time_dict, event_dict, test_ids,
                               batch_size=batch_size, device=device)
        cidx_b = concordance_index(
            base_test["time"].values,
            -np.asarray(base_test["risk"].values),
            base_test["event"].values
        )
        base_cidx_folds.append(float(cidx_b))
        base_tests_folds.append(base_test)
        print(f"[Fold {fold}] Baseline Test C-index: {cidx_b:.4f}")

        # per-fold KM curves + log-rank p-value for Baseline
        base_curves, base_p = km_curves_and_logrank(base_test)
        base_km_pvals.append(base_p)

        # Save per-fold KM curves (long format: time, survival, group)
        base_rows = []
        for grp in ["Low", "High"]:
            t, s = base_curves[grp]
            for ti, si in zip(t, s):
                base_rows.append({"time": float(ti), "survival": float(si), "group": grp})
        base_km_df = pd.DataFrame(base_rows)
            os.path.join(out_dir, f"baseline_km_fold_{fold}.csv"),
            index=False
        )

        # ---- Train Ours ----
        print(f"[Fold {fold}] Training Ours ...")
        ours_model = train_end2end_milcox(
            our_embeddings, time_dict, event_dict, tr_ids, va_ids,
            batch_size=batch_size, max_epochs=max_epochs,
            pool=pool, lr=lr, wd=wd, embed_dim=embed_dim, device=device, seed=seed
        )
        ours_test = predict_df(ours_model, our_embeddings, time_dict, event_dict, test_ids,
                               batch_size=batch_size, device=device)
        cidx_o = concordance_index(
            ours_test["time"].values,
            -np.asarray(ours_test["risk"].values),
            ours_test["event"].values
        )
        ours_cidx_folds.append(float(cidx_o))
        ours_tests_folds.append(ours_test)
        print(f"[Fold {fold}] Ours Test C-index: {cidx_o:.4f}")

        # per-fold KM curves + log-rank p-value for Ours
        ours_curves, ours_p = km_curves_and_logrank(ours_test)
        ours_km_pvals.append(ours_p)

        # Save per-fold KM curves (long format: time, survival, group)
        ours_rows = []
        for grp in ["Low", "High"]:
            t, s = ours_curves[grp]
            for ti, si in zip(t, s):
                ours_rows.append({"time": float(ti), "survival": float(si), "group": grp})
        ours_km_df = pd.DataFrame(ours_rows)
            os.path.join(out_dir, f"ours_km_fold_{fold}.csv"),
            index=False
        )

        # Save raw per-fold test preds as before

    # ---------------- Save metrics JSON ----------------
    results = {
        "fold_cindex": {
            "Baseline": base_cidx_folds,
            "Ours": ours_cidx_folds
        },
        "km_logrank_p": {
            "Baseline": base_km_pvals,
            "Ours": ours_km_pvals
        },
        "meta": {
            "n_ids": len(all_ids),
            "methods": ["Baseline", "Ours"],
            "n_folds": 5
        }
    }
    json_path = os.path.join(out_dir, "cox_cv5_results.json")
    with open(json_path, "w") as f:
    print(f"[CV] Saved metrics JSON → {json_path}")

    # ---------------- C-index boxplot ----------------
    plt.figure(figsize=(7.2, 4.6))
    plt.boxplot(
        [base_cidx_folds, ours_cidx_folds],
        labels=["Baseline", "Ours"],
        showmeans=False
    )
    plt.ylabel("C-index (higher is better)")
    plt.title("MIL–Cox — 5-fold Test C-index")
    plt.ylim(0.4, 1.0)
    for i, arr in enumerate([base_cidx_folds, ours_cidx_folds], start=1):
        plt.text(i, np.nanmean(arr)+0.02, f"{np.nanmean(arr):.3f}", ha="center", fontsize=10)
    plt.tight_layout()
    box_svg = os.path.join(out_dir, "cindex_boxplot.svg")
    box_png = os.path.join(out_dir, "cindex_boxplot.png")
    plt.close()
    print(f"[CV] Saved boxplot → {box_svg}")

    # return artifacts (no mean KM curve drawing)
    return {
        "json": json_path,
        "cindex_boxplot_svg": box_svg,
        "base_tests_folds": base_tests_folds,
        "ours_tests_folds": ours_tests_folds,
        "base_cidx_folds": base_cidx_folds,
        "ours_cidx_folds": ours_cidx_folds,
        "baseline_km_pvalues": base_km_pvals,
        "ours_km_pvalues": ours_km_pvals,
    }




artifacts = run_cv5_milcox(
    time_dict=time_dict,
    event_dict=event_dict,
    baseline_embeddings=baseline_embeddings,
    our_embeddings=our_embeddings,
    out_dir="cox_cv5_outputs",
    pool="attn",         # "attn" | "mean" | "max"
    embed_dim=64,
    lr=1e-3, wd=0.0,
    max_epochs=10,
    batch_size=64,
    seed=42
)
# print("Artifacts:", artifacts)


In [ ]:
# ============================================================
# Post-hoc KM plotting from saved CSVs
# - Reads base_tests_folds_{i}.csv and ours_tests_folds_{i}.csv
# - For each fold:
#     * splits patients by median risk (Low vs High)
#     * fits KM curves
#     * runs log-rank test
#     * plots KM curves with p-value in the title
# ============================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype']  = 42


def km_curves_and_logrank_from_df(df: pd.DataFrame):
    """
    Given a test DataFrame with columns ['time','event','risk'],
    split by median risk → KM curves (Low/High) + log-rank p-value.

    Returns:
        curves: dict with keys 'Low', 'High', each → (time_array, surv_array)
        p_value: float
    """
    risk = df["risk"].values
    time = df["time"].values
    event = df["event"].values

    med = np.median(risk)
    grp = np.where(risk >= med, "High", "Low")

    curves = {}
    km = KaplanMeierFitter()
    for label in ["Low", "High"]:
        m = (grp == label)
        if m.sum() < 2:
            # Not enough samples; return trivial flat curve
            curves[label] = (np.array([0.0, 1.0]), np.array([1.0, 1.0]))
            continue
        km.fit(durations=time[m], event_observed=event[m], label=label)
        t = km.survival_function_.index.values.astype(float)
        s = km.survival_function_[label].values.astype(float)
        # Ensure curve starts at (0, 1)
        if t[0] > 0:
            t = np.insert(t, 0, 0.0)
            s = np.insert(s, 0, 1.0)
        curves[label] = (t, s)

    # log-rank test between Low vs High
    low_mask = (grp == "Low")
    high_mask = (grp == "High")
    if low_mask.sum() >= 2 and high_mask.sum() >= 2:
        res = logrank_test(
            time[low_mask], time[high_mask],
            event_observed_A=event[low_mask],
            event_observed_B=event[high_mask]
        )
        p_value = float(res.p_value)
    else:
        p_value = float("nan")

    return curves, p_value


def plot_km_from_csv(
    out_dir: str = "cox_cv5_outputs",
    n_folds: int = 5,
    base_prefix: str = "base_tests_folds_",
    ours_prefix: str = "ours_tests_folds_",
):
    """
    Read per-fold test CSVs and draw KM plots for Baseline & Ours.

    For each fold i:
      - Reads:
          base_csv = f"{base_prefix}{i}.csv"
          ours_csv = f"{ours_prefix}{i}.csv"
      - Draws a figure with 2 subplots: Baseline (left), Ours (right)
      - Each subplot has Low vs High KM curves (median risk split)
      - Log-rank p-value is shown in the title
      - Saves SVG + PNG for that fold
    """
    base_pvals, ours_pvals = [], []

    os.makedirs(out_dir, exist_ok=True)

    for i in range(n_folds):
        base_csv = os.path.join(out_dir, f"{base_prefix}{i}.csv")
        ours_csv = os.path.join(out_dir, f"{ours_prefix}{i}.csv")

        if not (os.path.exists(base_csv) and os.path.exists(ours_csv)):
            print(f"[Fold {i}] Missing CSVs, skip: {base_csv} / {ours_csv}")
            continue

        base_df = pd.read_csv(base_csv)
        ours_df = pd.read_csv(ours_csv)

        # ---- Baseline KM ----
        base_curves, base_p = km_curves_and_logrank_from_df(base_df)
        base_pvals.append(base_p)

        # ---- Ours KM ----
        ours_curves, ours_p = km_curves_and_logrank_from_df(ours_df)
        ours_pvals.append(ours_p)

        # ---- Plot side-by-side for this fold ----
        fig, axes = plt.subplots(1, 2, figsize=(10, 4.0), sharey=True)
        colors = {"Low": "#1f77b4", "High": "#d62728"}  # blue / red

        # Baseline subplot
        ax = axes[0]
        for label in ["Low", "High"]:
            t, s = base_curves[label]
            ax.step(t, s, where="post", label=f"{label} risk", linewidth=2.0, color=colors[label])
        ax.set_title(
            f"Baseline — Fold {i}\nlog-rank p = {base_p:.2e}" if np.isfinite(base_p) else f"Baseline — Fold {i}\nlog-rank p = NA",
            fontsize=11
        )
        ax.set_xlabel("Time")
        ax.set_ylabel("Survival probability")
        ax.set_ylim(0.0, 1.0)
        ax.grid(False)
        ax.legend(loc="upper right", fontsize=8)

        # Ours subplot
        ax = axes[1]
        for label in ["Low", "High"]:
            t, s = ours_curves[label]
            ax.step(t, s, where="post", label=f"{label} risk", linewidth=2.0, color=colors[label])
        ax.set_title(
            f"Ours — Fold {i}\nlog-rank p = {ours_p:.2e}" if np.isfinite(ours_p) else f"Ours — Fold {i}\nlog-rank p = NA",
            fontsize=11
        )
        ax.set_xlabel("Time")
        ax.grid(False)
        ax.legend(loc="upper right", fontsize=8)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        plt.tight_layout()
        svg_path = os.path.join(out_dir, f"new_km_fold_{i}_baseline_vs_ours.svg")
        png_path = os.path.join(out_dir, f"new_km_fold_{i}_baseline_vs_ours.png")
        plt.close()

        print(f"[Fold {i}] Saved KM plot → {svg_path}")

    # Optional: print summary of p-values˜˜
    print("Baseline log-rank p-values per fold:", base_pvals)
    print("Ours log-rank p-values per fold:    ", ours_pvals)

    return {"baseline_p": base_pvals, "ours_p": ours_pvals}


# ============================================================
# Call this AFTER you have already run run_cv5_milcox(...)
# and generated base_tests_folds_*.csv / ours_tests_folds_*.csv
# ============================================================

_km_stats = plot_km_from_csv(out_dir="cox_cv5_outputs", n_folds=5)


In [ ]:
import seaborn as sns
import json
from scipy.stats import ranksums
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set font to Arial, large bold fonts for clarity, and ensure Illustrator compatibility/editability
plt.rcParams['font.family'] = 'Arial'    # use Arial font
plt.rcParams['svg.fonttype'] = 'none'    # don't convert text to path
plt.rcParams['pdf.fonttype'] = 42        # embed as TrueType for AI compatibility
plt.rcParams['ytick.labelsize'] = 10     # Increase y-tick font size
plt.rcParams['font.weight'] = 'bold'     # make text bold
plt.rcParams['axes.labelweight'] = 'bold'# bold axis labels
plt.rcParams['axes.titleweight'] = 'bold'# bold title

def draw_plot(results, out_dir='cox_cv5_outputs'):
    base_cidx_folds = results["fold_cindex"]["Baseline"]
    ours_cidx_folds = results["fold_cindex"]["Ours"]
    base_c = np.array(base_cidx_folds)
    ours_c = np.array(ours_cidx_folds)

    # Build DataFrame for seaborn
    df_plot = pd.DataFrame({
        "C-index": np.concatenate([base_c, ours_c]),
        "Method":  ["Baseline"] * len(base_c) + ["Ours"] * len(ours_c)
    })

    plt.figure(figsize=(6, 8))

    # Custom color palette
    palette = {
        "Baseline": '#E68B81',   # light blue
        "Ours":     "#B7B2D0",   # light orange
    }

    # Compute means and stds for error bars
    group_stats = df_plot.groupby("Method")["C-index"].agg(['mean', 'std']).reindex(["Baseline", "Ours"])
    means = group_stats["mean"].values
    stds = group_stats["std"].values

    # Draw the barplot with errorbars
    bar = plt.bar(
        x=["Baseline", "Ours"],
        height=means,
        yerr=stds,
        color=[palette["Baseline"], palette["Ours"]],
        capsize=8,
        edgecolor="black",
        width=0.5,
        linewidth=2
    )

    # Annotate mean value on top of each bar
    for i, rect in enumerate(bar):
        height = rect.get_height()
        plt.text(
            rect.get_x() + rect.get_width() / 2,
            height + 0.02,
            f"{height:.3f}",
            ha='center',
            va='bottom',
            fontsize=13,
            fontweight='bold'
        )

    # Overlay scatter points of individual values ("swarm"/strip)
    '''for i, method in enumerate(["Baseline", "Ours"]):
        method_values = df_plot[df_plot["Method"] == method]["C-index"].values
        x_jitter = np.random.normal(i, 0.05, size=len(method_values))
        plt.scatter(
            x_jitter,
            method_values,
            color='k',
            s=40,
            alpha=0.6,
            linewidth=0.7,
            edgecolors="white"
        )'''

    plt.xticks([0, 1], ["VirTues", "Haiku"], fontsize=14)
    plt.ylim(0, 1.0)
    plt.ylabel("C-index", fontsize=14)
    plt.yticks(fontsize=14)
    plt.title("Cross-validated C-index by method", fontsize=14)
    plt.grid(False)
    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    # ==== Increased boundary width below ====
    for spine in ['left', 'bottom']:
        ax.spines[spine].set_linewidth(2)  # Increased from default (usually 1) to 3

    bar_svg = os.path.join(out_dir, "cindex_barplot.svg")
    bar_png = os.path.join(out_dir, "cindex_barplot.png")
    plt.show()
    plt.close()

    # One-sided rank-sum test, alternative: ours > baseline
    stat, p_one_sided = ranksums(ours_c, base_c, alternative='greater')
    print("Wilcoxon rank-sum test (one-sided, Ours > Baseline):")
    print(f"  statistic = {stat:.3f}, one-sided p = {p_one_sided:.3g}")

    print(f"[CV] Saved barplot → {bar_svg}")

results = json.load(open("cox_cv5_outputs/cox_cv5_results.json", 'r'))

print(results)

draw_plot(results)